# 03 · NER compared: spaCy, GLiNER and Haiku

**Scope.** The same 343 articles as `02_explore_ner_de.ipynb`, three models side by
side. This is why Haiku was kept; only Haiku's mentions go further.

| Model | Kind | Labels |
|---|---|---|
| **spaCy** `de_core_news_lg` | classic, trained on German news | PER, ORG, LOC, MISC |
| **GLiNER** `urchade/gliner_multi-v2.1` | zero-shot, we name the labels | person, organization |
| **Claude Haiku 5.5** | prompted, tags the text inline | person, organization |

- **spaCy's LOC and MISC are kept as type `other`.** That lets us see whether a
  model tags *Bern* (meaning the government) as a place. Only person and
  organization enter the counts.
- **GLiNER reads at most 384 tokens at once.** Articles are cut into pieces of
  at most 150 words, at a sentence or line end where possible so that no name
  is split, and the positions are shifted back onto the full article.

Each model's output comes from `data/interim/`, saved under a hash of its
settings. Haiku's tags are read from the cache: this notebook makes no API call.


In [1]:
import pandas as pd

from mediapos.ner import gliner_mentions, haiku_mentions, spacy_mentions, tag_results
from mediapos.preprocess import event_archive, vote_articles
from mediapos.relevance import add_labels, kept_articles, load_review

_, _, archive = event_archive("A-631")
related = kept_articles(add_labels(vote_articles(archive, "de")), load_review())
texts = dict(zip(related["id"], related["text"]))
print(f"{len(related)} articles, {related['text'].str.split().str.len().sum():,} words")

343 articles, 234,772 words


In [2]:
spacy_found = spacy_mentions(related)
gliner_found = gliner_mentions(related)
haiku_found = haiku_mentions(related, tag_results())
print(f"spaCy {len(spacy_found):,}, GLiNER {len(gliner_found):,}, Haiku {len(haiku_found):,} mentions")

spaCy 16,765, GLiNER 9,504, Haiku 9,234 mentions


### Checks

Every stored position must point at the stored mention text, and the per-article
dictionaries must be an exact tally of the mention rows.

In [3]:
mentions = pd.concat([spacy_found, gliner_found, haiku_found], ignore_index=True)
assert all(texts[r.id][r.start:r.end] == r.mention for r in mentions.itertuples()), "a position is off"
assert mentions.groupby("model")["id"].nunique().max() <= len(related)

named = mentions[mentions["type"] != "other"]


def entity_counts(model: str) -> dict[str, dict[str, int]]:
    """{article id: {mention: count}} for one model, persons and organizations only."""
    found = named[named["model"] == model]
    return {i: g["mention"].value_counts().to_dict() for i, g in found.groupby("id")}


for model in ("spacy", "gliner", "haiku"):
    tally = sum(sum(d.values()) for d in entity_counts(model).values())
    assert tally == (named["model"] == model).sum()
print("positions and tallies check out")

positions and tallies check out


### First look

In [4]:
(named.groupby("model")
      .agg(mentions=("id", "size"),
           persons=("type", lambda t: (t == "person").sum()),
           organizations=("type", lambda t: (t == "organization").sum()),
           distinct_names=("mention", "nunique"),
           articles_with_none=("id", lambda i: len(related) - i.nunique()))
      .assign(per_article=lambda t: (t.mentions / len(related)).round(1)))

,mentions,persons,organizations,distinct_names,articles_with_none,per_article
model,,,,,,
gliner,9504,3098,6406,1566,0,27.7
haiku,9234,3215,6019,909,1,26.9
spacy,8979,3410,5569,1433,0,26.2


In [5]:
def side_by_side(article_id: str) -> pd.DataFrame:
    """Mention counts in one article, one column per model."""
    one = named[named["id"] == article_id]
    return (one.pivot_table(index="mention", columns="model", values="start", aggfunc="count", fill_value=0)
               .reindex(columns=["spacy", "gliner", "haiku"], fill_value=0)
               .sort_values(["haiku", "spacy", "gliner"], ascending=False))


example = related.sample(1, random_state=7).iloc[0]
print(f"{example['id']}  {example['medium_name']}")
side_by_side(example["id"])

40554906  Blick


model,spacy,gliner,haiku
mention,,,
Dettling,7,7,7
SVP,4,4,6
Zimmermann,0,4,4
Bundesrat,0,3,3
EU,2,2,2
Laura Zimmermann,2,2,2
Marcel Dettling,2,2,2
BLICK,0,0,2
Operation Libero,0,0,2
